In [1]:
"""
XOR exercise: why do we need a nonlinearity (and hidden neurons)?

Pure numpy, hand-written backprop, in the spirit of Karpathy's makemore.
Run:  python xor_exercise.py
"""
import numpy as np

# The four XOR points. Output is 1 when the inputs differ.
X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = np.array([[0.], [1.], [1.], [0.]])


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


ACTS = {
    # name: (forward, derivative expressed in terms of the OUTPUT h)
    "tanh":     (np.tanh,        lambda h: 1 - h**2),   # same (1 - tanh^2) as in the lecture
    "identity": (lambda a: a,    lambda h: np.ones_like(h)),
}


def train(hidden=2, act="tanh", steps=5000, lr=0.5, seed=0):
    rng = np.random.default_rng(seed)
    f, df = ACTS[act]
    W1 = rng.normal(0, 1, (2, hidden)); b1 = np.zeros(hidden)
    W2 = rng.normal(0, 1, (hidden, 1)); b2 = np.zeros(1)

    for step in range(steps + 1):
        # ---- forward ----
        pre = X @ W1 + b1
        h = f(pre)
        p = sigmoid(h @ W2 + b2)
        loss = -np.mean(Y * np.log(p + 1e-9) + (1 - Y) * np.log(1 - p + 1e-9))

        if step % 1000 == 0:
            print(f"  step {step:5d}  loss {loss:.4f}")

        # ---- backward (chain rule, layer by layer) ----
        dz = (p - Y) / len(X)          # grad of BCE wrt output logit
        dW2 = h.T @ dz
        db2 = dz.sum(0)
        dh = dz @ W2.T
        dpre = dh * df(h)              # <- where the activation's derivative enters
        dW1 = X.T @ dpre
        db1 = dpre.sum(0)

        # ---- update ----
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2

    return W1, b1, W2, b2, h, p


def report(p):
    print("  inputs -> prediction (target)")
    for x, pi, y in zip(X, p, Y):
        print(f"   {x.astype(int)} -> {pi[0]:.3f}   ({int(y[0])})")


print("=" * 60)
print("EXPERIMENT 1: two layers, NO activation function")
print("=" * 60)
W1, b1, W2, b2, h, p = train(hidden=2, act="identity")
report(p)
print("\n  Two stacked linear layers collapse into one:")
print("  W1 @ W2 =", (W1 @ W2).ravel(), "  <- just ONE weight per input.")
print("  A single line can't separate XOR, so loss stays near ln(2)=0.693.\n")

print("=" * 60)
print("EXPERIMENT 2: two layers, tanh in between")
print("=" * 60)
W1, b1, W2, b2, h, p = train(hidden=2, act="tanh")
report(p)
print("\n  What the 2 hidden neurons output for each input (the new 'space'):")
for x, hi in zip(X, h):
    print(f"   {x.astype(int)} -> h = [{hi[0]:+.2f}, {hi[1]:+.2f}]")
print("\n  The hidden layer has bent the 4 points so that a single line")
print("  (the output neuron) can now separate them.\n")

print("=" * 60)
print("THINGS TO TRY")
print("=" * 60)
print("""
1. Set hidden=1 with tanh. Does it still solve XOR? Why not?
2. Try hidden=2 with several seeds. Does it ever get stuck?
   Print pre-activations: are the tanh units saturated (|pre| large)?
   That is the dead-gradient problem from episode 4.
3. Multiply the initial W1 by 10 and rerun. Watch the loss barely move
   at first: saturated tanh => (1 - h^2) ~ 0 => gradient ~ 0.
4. Add a ReLU entry to ACTS and compare.
""")

EXPERIMENT 1: two layers, NO activation function
  step     0  loss 0.7019
  step  1000  loss 0.6931
  step  2000  loss 0.6931
  step  3000  loss 0.6931
  step  4000  loss 0.6931
  step  5000  loss 0.6931
  inputs -> prediction (target)
   [0 0] -> 0.500   (0)
   [0 1] -> 0.500   (1)
   [1 0] -> 0.500   (1)
   [1 1] -> 0.500   (0)

  Two stacked linear layers collapse into one:
  W1 @ W2 = [-3.35452152e-16 -1.49880108e-15]   <- just ONE weight per input.
  A single line can't separate XOR, so loss stays near ln(2)=0.693.

EXPERIMENT 2: two layers, tanh in between
  step     0  loss 0.7027
  step  1000  loss 0.0079
  step  2000  loss 0.0035
  step  3000  loss 0.0022
  step  4000  loss 0.0016
  step  5000  loss 0.0013
  inputs -> prediction (target)
   [0 0] -> 0.001   (0)
   [0 1] -> 0.998   (1)
   [1 0] -> 0.998   (1)
   [1 1] -> 0.001   (0)

  What the 2 hidden neurons output for each input (the new 'space'):
   [0 0] -> h = [+0.96, -0.95]
   [0 1] -> h = [+1.00, +0.96]
   [1 0] -> h 